# Lab 9: Running Mandelbrot on an AMD ROCm Cluster

[Chinese version](Lab9_Mandelbrot_AMD_ROCm_ZH.ipynb) | English

This notebook does two things:

1. **Original C project**: if the container has `gcc` / `mpicc` / `mpirun`, compile and run the original serial and MPI+OpenMP static/dynamic versions.
2. **GPU port**: port the same Mandelbrot algorithm to the AMD GPU with PyTorch ROCm, measure time, save images and compare with the C serial output.

> Note: a Jupyter container is normally a single node, so the MPI part here is multi-process on one container, not multi-node. The two-node cluster result remains the one in the report.

Environment: Python 3.12, PyTorch 2.9.1+rocm7.13, HIP/ROCm 7.13, Radeon 8060S.


## 0. Environment check

Check Python, compilers, MPI and GPU availability first.


In [ ]:
import os, sys, platform, shutil, subprocess, time, json, re, hashlib
from pathlib import Path

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())
print()
for cmd in ["gcc", "mpicc", "mpirun", "hipcc", "rocminfo"]:
    print(f"{cmd:8s}: {shutil.which(cmd)}")

print()
try:
    import torch
    print("torch      :", torch.__version__)
    print("torch HIP  :", torch.version.hip)
    print("CUDA API   :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU        :", torch.cuda.get_device_name(0))
        p = torch.cuda.get_device_properties(0)
        print("GPU memory : %.1f GB" % (p.total_memory / 1e9))
        if hasattr(p, "gcnArchName"):
            print("gcnArchName:", p.gcnArchName)
except Exception as e:
    print("torch error:", repr(e))

## 1. Install MPI and OpenMP if missing

The cell below checks `gcc`, `mpicc` and `mpirun`. If they are missing it tries
to install them automatically:

- `apt-get` with passwordless `sudo` (or as root);
- otherwise `conda` / `mamba` / `micromamba` from conda-forge.

If the container does not allow either method, the cell prints the commands to
run manually. OpenMP is part of GCC; there is no separate OpenMP package.

After a successful installation, the build cell below will compile and run the
static and dynamic 2x2 configurations.


In [ ]:
import os, sys, shutil, subprocess, platform, importlib

# Set this to False if you do not want the notebook to try installing packages.
INSTALL_MPI = True

def run(cmd, timeout=900):
    print("$", " ".join(str(x) for x in cmd))
    env = os.environ.copy()
    env["DEBIAN_FRONTEND"] = "noninteractive"
    try:
        p = subprocess.run(cmd, text=True, capture_output=True, env=env, timeout=timeout)
    except Exception as e:
        print("FAILED:", repr(e))
        return None
    out = (p.stdout or "") + (p.stderr or "")
    if out:
        print(out[-3000:])
    return p

def sudo_n_ok():
    sudo = shutil.which("sudo")
    if not sudo:
        return False
    p = run([sudo, "-n", "true"], timeout=15)
    return bool(p and p.returncode == 0)

def have_toolchain():
    return bool(shutil.which("gcc") and shutil.which("mpicc") and shutil.which("mpirun"))

print("before:", {k: shutil.which(k) for k in ["gcc", "mpicc", "mpirun"]})

if not have_toolchain() and INSTALL_MPI:
    apt = shutil.which("apt-get")
    if apt:
        if os.geteuid() == 0:
            prefix = []
        elif sudo_n_ok():
            prefix = ["sudo", "-n"]
        else:
            prefix = None
        if prefix is not None:
            print("\nTrying apt-get (build-essential + OpenMPI)...")
            run(prefix + ["apt-get", "update"])
            run(prefix + ["apt-get", "install", "-y",
                          "build-essential", "libopenmpi-dev", "openmpi-bin"])

    if not have_toolchain():
        mgr = next((shutil.which(x) for x in ["conda", "mamba", "micromamba"]
                    if shutil.which(x)), None)
        if mgr:
            machine = platform.machine().lower()
            if machine in ("aarch64", "arm64"):
                compilers = ["gcc_linux-aarch64", "gxx_linux-aarch64"]
            else:
                compilers = ["gcc_linux-64", "gxx_linux-64"]
            pkgs = ["openmpi"]
            if not shutil.which("gcc"):
                pkgs += compilers
            print("\nTrying conda-forge OpenMPI...")
            run([mgr, "install", "-y", "-c", "conda-forge", *pkgs], timeout=1200)

# Make a conda-installed toolchain visible in this kernel.
for candidate in [os.path.join(sys.prefix, "bin"),
                  os.path.join(os.environ.get("CONDA_PREFIX", ""), "bin")]:
    if candidate and os.path.isdir(candidate) and candidate not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = candidate + os.pathsep + os.environ["PATH"]

importlib.invalidate_caches()
print("after :", {k: shutil.which(k) for k in ["gcc", "mpicc", "mpirun"]})

if not have_toolchain():
    print("\nMPI is still missing. Run one of these in a terminal, then restart the kernel:")
    print("  sudo apt-get update && sudo apt-get install -y build-essential libopenmpi-dev openmpi-bin")
    print("  conda install -y -c conda-forge openmpi gcc_linux-64 gxx_linux-64")
else:
    print("\nMPI/OpenMP toolchain is ready.")


## 2. Materialise the original C sources

The notebook embeds the final `mandelbrot_serial.c` and `mandelbrot_mpi_omp.c`, so no network clone is needed. The static variant is generated by replacing the dynamic pragma.


In [ ]:
import base64
from pathlib import Path

WORK = Path("lab9_amd_work")
WORK.mkdir(exist_ok=True)

SERIAL_B64 = "I2luY2x1ZGUgPHN0ZGlvLmg+CiNpbmNsdWRlIDxzdGRsaWIuaD4KI2luY2x1ZGUgPHN0ZGludC5oPgojaW5jbHVkZSA8aW50dHlwZXMuaD4KI2luY2x1ZGUgPG1hdGguaD4KI2luY2x1ZGUgPHRpbWUuaD4KCiNkZWZpbmUgREVGQVVMVF9XSURUSCAgICA4MDAKI2RlZmluZSBERUZBVUxUX0hFSUdIVCAgIDYwMAojZGVmaW5lIERFRkFVTFRfTUFYX0lURVIgMTAwMAoKc3RhdGljIGRvdWJsZSBub3dfc2Vjb25kcyh2b2lkKQp7CiAgICBzdHJ1Y3QgdGltZXNwZWMgdHM7CiAgICB0aW1lc3BlY19nZXQoJnRzLCBUSU1FX1VUQyk7CiAgICByZXR1cm4gKGRvdWJsZSl0cy50dl9zZWMgKyAxZS05ICogKGRvdWJsZSl0cy50dl9uc2VjOwp9CgpzdGF0aWMgdWludDY0X3QgZm52MWEoY29uc3QgdW5zaWduZWQgY2hhciAqZGF0YSwgc2l6ZV90IGxlbikKewogICAgdWludDY0X3QgaCA9IDE0Njk1OTgxMDM5MzQ2NjU2MDN1bGw7CiAgICBmb3IgKHNpemVfdCBpID0gMDsgaSA8IGxlbjsgKytpKSB7CiAgICAgICAgaCBePSAodWludDY0X3QpZGF0YVtpXTsKICAgICAgICBoICo9IDEwOTk1MTE2MjgyMTF1bGw7CiAgICB9CiAgICByZXR1cm4gaDsKfQoKc3RhdGljIGludCB3cml0ZV9wcG0oY29uc3QgY2hhciAqcGF0aCwgY29uc3QgdW5zaWduZWQgY2hhciAqcGl4ZWxzLAogICAgICAgICAgICAgICAgICAgICBpbnQgd2lkdGgsIGludCBoZWlnaHQpCnsKICAgIEZJTEUgKmYgPSBmb3BlbihwYXRoLCAid2IiKTsKICAgIGlmICghZikgewogICAgICAgIHBlcnJvcihwYXRoKTsKICAgICAgICByZXR1cm4gMDsKICAgIH0KICAgIGZwcmludGYoZiwgIlA2XG4lZCAlZFxuMjU1XG4iLCB3aWR0aCwgaGVpZ2h0KTsKCiAgICBzaXplX3QgYnl0ZXMgPSAoc2l6ZV90KXdpZHRoICogKHNpemVfdCloZWlnaHQgKiAzdTsKICAgIGlmIChmd3JpdGUocGl4ZWxzLCAxLCBieXRlcywgZikgIT0gYnl0ZXMpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgInNob3J0IHdyaXRlIHRvICVzXG4iLCBwYXRoKTsKICAgICAgICBmY2xvc2UoZik7CiAgICAgICAgcmV0dXJuIDA7CiAgICB9CiAgICBmY2xvc2UoZik7CiAgICByZXR1cm4gMTsKfQoKaW50IG1haW4oaW50IGFyZ2MsIGNoYXIgKiphcmd2KQp7CiAgICBpbnQgd2lkdGggICAgPSAoYXJnYyA+IDEpID8gYXRvaShhcmd2WzFdKSA6IERFRkFVTFRfV0lEVEg7CiAgICBpbnQgaGVpZ2h0ICAgPSAoYXJnYyA+IDIpID8gYXRvaShhcmd2WzJdKSA6IERFRkFVTFRfSEVJR0hUOwogICAgaW50IG1heF9pdGVyID0gKGFyZ2MgPiAzKSA/IGF0b2koYXJndlszXSkgOiBERUZBVUxUX01BWF9JVEVSOwogICAgY29uc3QgY2hhciAqb3V0ID0gKGFyZ2MgPiA0KSA/IGFyZ3ZbNF0gOiAibWFuZGVsYnJvdC5wcG0iOwoKICAgIGlmICh3aWR0aCA8IDE2IHx8IGhlaWdodCA8IDE2IHx8IG1heF9pdGVyIDwgMSkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAidXNhZ2U6ICVzIFt3aWR0aF0gW2hlaWdodF0gW21heF9pdGVyXSBbb3V0LnBwbV1cbiIsIGFyZ3ZbMF0pOwogICAgICAgIHJldHVybiAxOwogICAgfQoKICAgIHVuc2lnbmVkIGNoYXIgKnBpeGVscyA9IG1hbGxvYygoc2l6ZV90KXdpZHRoICogKHNpemVfdCloZWlnaHQgKiAzdSk7CiAgICBpZiAoIXBpeGVscykgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAib3V0IG9mIG1lbW9yeVxuIik7CiAgICAgICAgcmV0dXJuIDE7CiAgICB9CgogICAgY29uc3QgZG91YmxlIHhtaW4gPSAtMi4xMDsKICAgIGNvbnN0IGRvdWJsZSB4bWF4ID0gIDAuNzA7CiAgICBjb25zdCBkb3VibGUgeHNwYW4gPSB4bWF4IC0geG1pbjsKICAgIGNvbnN0IGRvdWJsZSB5c3BhbiA9IHhzcGFuICogKGRvdWJsZSloZWlnaHQgLyAoZG91YmxlKXdpZHRoOwogICAgY29uc3QgZG91YmxlIHltaW4gPSAtMC41ICogeXNwYW47CiAgICBjb25zdCBkb3VibGUgeW1heCA9ICAwLjUgKiB5c3BhbjsKCiAgICBkb3VibGUgdDAgPSBub3dfc2Vjb25kcygpOwoKICAgIGZvciAoaW50IHB5ID0gMDsgcHkgPCBoZWlnaHQ7ICsrcHkpIHsKICAgICAgICBkb3VibGUgY2kgPSB5bWF4IC0gKHltYXggLSB5bWluKSAqICgoZG91YmxlKXB5ICsgMC41KSAvIChkb3VibGUpaGVpZ2h0OwoKICAgICAgICBmb3IgKGludCBweCA9IDA7IHB4IDwgd2lkdGg7ICsrcHgpIHsKICAgICAgICAgICAgZG91YmxlIGNyID0geG1pbiArICh4bWF4IC0geG1pbikgKiAoKGRvdWJsZSlweCArIDAuNSkgLyAoZG91YmxlKXdpZHRoOwoKICAgICAgICAgICAgZG91YmxlIHpyID0gMC4wOwogICAgICAgICAgICBkb3VibGUgemkgPSAwLjA7CiAgICAgICAgICAgIGludCBpdGVyID0gMDsKCiAgICAgICAgICAgIHdoaWxlIChpdGVyIDwgbWF4X2l0ZXIgJiYgenIgKiB6ciArIHppICogemkgPD0gNC4wKSB7CiAgICAgICAgICAgICAgICBkb3VibGUgbmV4dF9yID0genIgKiB6ciAtIHppICogemkgKyBjcjsKICAgICAgICAgICAgICAgIGRvdWJsZSBuZXh0X2kgPSAyLjAgKiB6ciAqIHppICsgY2k7CiAgICAgICAgICAgICAgICB6ciA9IG5leHRfcjsKICAgICAgICAgICAgICAgIHppID0gbmV4dF9pOwogICAgICAgICAgICAgICAgKytpdGVyOwogICAgICAgICAgICB9CgogICAgICAgICAgICB1bnNpZ25lZCBjaGFyIHIsIGcsIGI7CiAgICAgICAgICAgIGlmIChpdGVyID09IG1heF9pdGVyKSB7CiAgICAgICAgICAgICAgICByID0gZyA9IGIgPSAwOwogICAgICAgICAgICB9IGVsc2UgewogICAgICAgICAgICAgICAgZG91YmxlIHQgPSBzcXJ0KChkb3VibGUpaXRlciAvIDIwMC4wKTsKICAgICAgICAgICAgICAgIGlmICh0ID4gMS4wKQogICAgICAgICAgICAgICAgICAgIHQgPSAxLjA7CiAgICAgICAgICAgICAgICBkb3VibGUgcnIgPSBmbWluKDEuMCwgMi4wICogdCk7CiAgICAgICAgICAgICAgICBkb3VibGUgZ2cgPSBmbWluKDEuMCwgMi4wICogdCAtIDAuNSk7CiAgICAgICAgICAgICAgICBkb3VibGUgYmIgPSBmbWluKDEuMCwgMS4wIC0gMi4wICogdCk7CiAgICAgICAgICAgICAgICByID0gKHVuc2lnbmVkIGNoYXIpKDI1NS4wICogcnIpOwogICAgICAgICAgICAgICAgZyA9ICh1bnNpZ25lZCBjaGFyKSgyNTUuMCAqIGdnKTsKICAgICAgICAgICAgICAgIGIgPSAodW5zaWduZWQgY2hhcikoMjU1LjAgKiBiYik7CiAgICAgICAgICAgIH0KCiAgICAgICAgICAgIHNpemVfdCBpZHggPSAoKHNpemVfdClweSAqIChzaXplX3Qpd2lkdGggKyAoc2l6ZV90KXB4KSAqIDN1OwogICAgICAgICAgICBwaXhlbHNbaWR4ICsgMF0gPSByOwogICAgICAgICAgICBwaXhlbHNbaWR4ICsgMV0gPSBnOwogICAgICAgICAgICBwaXhlbHNbaWR4ICsgMl0gPSBiOwogICAgICAgIH0KICAgIH0KCiAgICBkb3VibGUgZWxhcHNlZCA9IG5vd19zZWNvbmRzKCkgLSB0MDsKICAgIHVpbnQ2NF90IGhhc2ggPSBmbnYxYShwaXhlbHMsIChzaXplX3Qpd2lkdGggKiAoc2l6ZV90KWhlaWdodCAqIDN1KTsKCiAgICBpZiAoIXdyaXRlX3BwbShvdXQsIHBpeGVscywgd2lkdGgsIGhlaWdodCkpIHsKICAgICAgICBmcmVlKHBpeGVscyk7CiAgICAgICAgcmV0dXJuIDE7CiAgICB9CgogICAgcHJpbnRmKCJNYW5kZWxicm90ICVkeCVkIG1heF9pdGVyPSVkXG4iLCB3aWR0aCwgaGVpZ2h0LCBtYXhfaXRlcik7CiAgICBwcmludGYoInNlcmlhbCB0aW1lIDogJS4zZiBzXG4iLCBlbGFwc2VkKTsKICAgIHByaW50ZigiRk5WLTFhIGhhc2ggOiAlMDE2IiBQUkl4NjQgIlxuIiwgaGFzaCk7CiAgICBwcmludGYoIm91dHB1dCAgICAgIDogJXNcbiIsIG91dCk7CgogICAgZnJlZShwaXhlbHMpOwogICAgcmV0dXJuIDA7Cn0K"
DYNAMIC_B64 = "I2luY2x1ZGUgPG1waS5oPgojaW5jbHVkZSA8c3RkaW8uaD4KI2luY2x1ZGUgPHN0ZGxpYi5oPgojaW5jbHVkZSA8c3RkaW50Lmg+CiNpbmNsdWRlIDxpbnR0eXBlcy5oPgojaW5jbHVkZSA8bWF0aC5oPgojaW5jbHVkZSA8b21wLmg+CgojZGVmaW5lIERFRkFVTFRfV0lEVEggICAgODAwCiNkZWZpbmUgREVGQVVMVF9IRUlHSFQgICA2MDAKI2RlZmluZSBERUZBVUxUX01BWF9JVEVSIDEwMDAKCnN0YXRpYyB1aW50NjRfdCBmbnYxYShjb25zdCB1bnNpZ25lZCBjaGFyICpkYXRhLCBzaXplX3QgbGVuKQp7CiAgICB1aW50NjRfdCBoID0gMTQ2OTU5ODEwMzkzNDY2NTYwM3VsbDsKICAgIGZvciAoc2l6ZV90IGkgPSAwOyBpIDwgbGVuOyArK2kpIHsKICAgICAgICBoIF49ICh1aW50NjRfdClkYXRhW2ldOwogICAgICAgIGggKj0gMTA5OTUxMTYyODIxMXVsbDsKICAgIH0KICAgIHJldHVybiBoOwp9CgpzdGF0aWMgaW50IHdyaXRlX3BwbShjb25zdCBjaGFyICpwYXRoLCBjb25zdCB1bnNpZ25lZCBjaGFyICpwaXhlbHMsCiAgICAgICAgICAgICAgICAgICAgIGludCB3aWR0aCwgaW50IGhlaWdodCkKewogICAgRklMRSAqZiA9IGZvcGVuKHBhdGgsICJ3YiIpOwogICAgaWYgKCFmKSB7CiAgICAgICAgcGVycm9yKHBhdGgpOwogICAgICAgIHJldHVybiAwOwogICAgfQogICAgZnByaW50ZihmLCAiUDZcbiVkICVkXG4yNTVcbiIsIHdpZHRoLCBoZWlnaHQpOwoKICAgIHNpemVfdCBieXRlcyA9IChzaXplX3Qpd2lkdGggKiAoc2l6ZV90KWhlaWdodCAqIDN1OwogICAgaWYgKGZ3cml0ZShwaXhlbHMsIDEsIGJ5dGVzLCBmKSAhPSBieXRlcykgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAic2hvcnQgd3JpdGUgdG8gJXNcbiIsIHBhdGgpOwogICAgICAgIGZjbG9zZShmKTsKICAgICAgICByZXR1cm4gMDsKICAgIH0KICAgIGZjbG9zZShmKTsKICAgIHJldHVybiAxOwp9CgovKiBSZW5kZXIgbnJvd3Mgcm93cyBzdGFydGluZyBhdCB5MCBpbnRvIGEgY29tcGFjdCBSR0IgYnVmZmVyLiAqLwpzdGF0aWMgdm9pZCByZW5kZXJfcm93cyhpbnQgeTAsIGludCBucm93cywgaW50IHdpZHRoLCBpbnQgaGVpZ2h0LAogICAgICAgICAgICAgICAgICAgICAgICBpbnQgbWF4X2l0ZXIsIHVuc2lnbmVkIGNoYXIgKm91dCkKewogICAgY29uc3QgZG91YmxlIHhtaW4gPSAtMi4xMDsKICAgIGNvbnN0IGRvdWJsZSB4bWF4ID0gIDAuNzA7CiAgICBjb25zdCBkb3VibGUgeHNwYW4gPSB4bWF4IC0geG1pbjsKICAgIGNvbnN0IGRvdWJsZSB5c3BhbiA9IHhzcGFuICogKGRvdWJsZSloZWlnaHQgLyAoZG91YmxlKXdpZHRoOwogICAgY29uc3QgZG91YmxlIHltaW4gPSAtMC41ICogeXNwYW47CiAgICBjb25zdCBkb3VibGUgeW1heCA9ICAwLjUgKiB5c3BhbjsKICAgICNwcmFnbWEgb21wIHBhcmFsbGVsIGZvciBzY2hlZHVsZShkeW5hbWljLCA4KQogICAgZm9yIChpbnQgaiA9IDA7IGogPCBucm93czsgKytqKSB7CiAgICAgICAgaW50IHB5ID0geTAgKyBqOwogICAgICAgIGRvdWJsZSBjaSA9IHltYXggLSAoeW1heCAtIHltaW4pICogKChkb3VibGUpcHkgKyAwLjUpIC8gKGRvdWJsZSloZWlnaHQ7CgogICAgICAgIGZvciAoaW50IHB4ID0gMDsgcHggPCB3aWR0aDsgKytweCkgewogICAgICAgICAgICBkb3VibGUgY3IgPSB4bWluICsgKHhtYXggLSB4bWluKSAqICgoZG91YmxlKXB4ICsgMC41KSAvIChkb3VibGUpd2lkdGg7CgogICAgICAgICAgICBkb3VibGUgenIgPSAwLjA7CiAgICAgICAgICAgIGRvdWJsZSB6aSA9IDAuMDsKICAgICAgICAgICAgaW50IGl0ZXIgPSAwOwoKICAgICAgICAgICAgd2hpbGUgKGl0ZXIgPCBtYXhfaXRlciAmJiB6ciAqIHpyICsgemkgKiB6aSA8PSA0LjApIHsKICAgICAgICAgICAgICAgIGRvdWJsZSBuZXh0X3IgPSB6ciAqIHpyIC0gemkgKiB6aSArIGNyOwogICAgICAgICAgICAgICAgZG91YmxlIG5leHRfaSA9IDIuMCAqIHpyICogemkgKyBjaTsKICAgICAgICAgICAgICAgIHpyID0gbmV4dF9yOwogICAgICAgICAgICAgICAgemkgPSBuZXh0X2k7CiAgICAgICAgICAgICAgICArK2l0ZXI7CiAgICAgICAgICAgIH0KCiAgICAgICAgICAgIHVuc2lnbmVkIGNoYXIgciwgZywgYjsKICAgICAgICAgICAgaWYgKGl0ZXIgPT0gbWF4X2l0ZXIpIHsKICAgICAgICAgICAgICAgIHIgPSBnID0gYiA9IDA7CiAgICAgICAgICAgIH0gZWxzZSB7CiAgICAgICAgICAgICAgICBkb3VibGUgdCA9IHNxcnQoKGRvdWJsZSlpdGVyIC8gMjAwLjApOwogICAgICAgICAgICAgICAgaWYgKHQgPiAxLjApCiAgICAgICAgICAgICAgICAgICAgdCA9IDEuMDsKICAgICAgICAgICAgICAgIGRvdWJsZSByciA9IGZtaW4oMS4wLCAyLjAgKiB0KTsKICAgICAgICAgICAgICAgIGRvdWJsZSBnZyA9IGZtaW4oMS4wLCAyLjAgKiB0IC0gMC41KTsKICAgICAgICAgICAgICAgIGRvdWJsZSBiYiA9IGZtaW4oMS4wLCAxLjAgLSAyLjAgKiB0KTsKICAgICAgICAgICAgICAgIHIgPSAodW5zaWduZWQgY2hhcikoMjU1LjAgKiBycik7CiAgICAgICAgICAgICAgICBnID0gKHVuc2lnbmVkIGNoYXIpKDI1NS4wICogZ2cpOwogICAgICAgICAgICAgICAgYiA9ICh1bnNpZ25lZCBjaGFyKSgyNTUuMCAqIGJiKTsKICAgICAgICAgICAgfQoKICAgICAgICAgICAgc2l6ZV90IGlkeCA9ICgoc2l6ZV90KWogKiAoc2l6ZV90KXdpZHRoICsgKHNpemVfdClweCkgKiAzdTsKICAgICAgICAgICAgb3V0W2lkeCArIDBdID0gcjsKICAgICAgICAgICAgb3V0W2lkeCArIDFdID0gZzsKICAgICAgICAgICAgb3V0W2lkeCArIDJdID0gYjsKICAgICAgICB9CiAgICB9Cn0KCmludCBtYWluKGludCBhcmdjLCBjaGFyICoqYXJndikKewogICAgaW50IHJhbmssIHNpemU7CgogICAgTVBJX0luaXQoJmFyZ2MsICZhcmd2KTsKICAgIE1QSV9Db21tX3JhbmsoTVBJX0NPTU1fV09STEQsICZyYW5rKTsKICAgIE1QSV9Db21tX3NpemUoTVBJX0NPTU1fV09STEQsICZzaXplKTsKCiAgICBpbnQgd2lkdGggICAgPSBERUZBVUxUX1dJRFRIOwogICAgaW50IGhlaWdodCAgID0gREVGQVVMVF9IRUlHSFQ7CiAgICBpbnQgbWF4X2l0ZXIgPSBERUZBVUxUX01BWF9JVEVSOwogICAgY29uc3QgY2hhciAqb3V0ID0gIm1waV9tYW5kZWxicm90LnBwbSI7CgogICAgaWYgKHJhbmsgPT0gMCkgewogICAgICAgIGlmIChhcmdjID4gMSkgd2lkdGggICAgPSBhdG9pKGFyZ3ZbMV0pOwogICAgICAgIGlmIChhcmdjID4gMikgaGVpZ2h0ICAgPSBhdG9pKGFyZ3ZbMl0pOwogICAgICAgIGlmIChhcmdjID4gMykgbWF4X2l0ZXIgPSBhdG9pKGFyZ3ZbM10pOwogICAgICAgIGlmIChhcmdjID4gNCkgb3V0ICAgICAgPSBhcmd2WzRdOwogICAgfQoKICAgIE1QSV9CY2FzdCgmd2lkdGgsIDEsIE1QSV9JTlQsIDAsIE1QSV9DT01NX1dPUkxEKTsKICAgIE1QSV9CY2FzdCgmaGVpZ2h0LCAxLCBNUElfSU5ULCAwLCBNUElfQ09NTV9XT1JMRCk7CiAgICBNUElfQmNhc3QoJm1heF9pdGVyLCAxLCBNUElfSU5ULCAwLCBNUElfQ09NTV9XT1JMRCk7CgogICAgaWYgKHdpZHRoIDwgMTYgfHwgaGVpZ2h0IDwgMTYgfHwgbWF4X2l0ZXIgPCAxKSB7CiAgICAgICAgaWYgKHJhbmsgPT0gMCkKICAgICAgICAgICAgZnByaW50ZihzdGRlcnIsICJ1c2FnZTogJXMgW3dpZHRoXSBbaGVpZ2h0XSBbbWF4X2l0ZXJdIFtvdXQucHBtXVxuIiwKICAgICAgICAgICAgICAgICAgICBhcmd2WzBdKTsKICAgICAgICBNUElfRmluYWxpemUoKTsKICAgICAgICByZXR1cm4gMTsKICAgIH0KCiAgICAvKiBEaXZpZGUgdGhlIHJvd3MgYmV0d2VlbiB0aGUgcmFua3MuICovCiAgICBpbnQgYmFzZSA9IGhlaWdodCAvIHNpemU7CiAgICBpbnQgcmVtICA9IGhlaWdodCAlIHNpemU7CiAgICBpbnQgbG9jYWxfaCA9IGJhc2UgKyAocmFuayA8IHJlbSA/IDEgOiAwKTsKICAgIGludCB5MCA9IHJhbmsgKiBiYXNlICsgKHJhbmsgPCByZW0gPyByYW5rIDogcmVtKTsKCiAgICBzaXplX3QgbG9jYWxfYnl0ZXMgPSAoc2l6ZV90KWxvY2FsX2ggKiAoc2l6ZV90KXdpZHRoICogM3U7CiAgICB1bnNpZ25lZCBjaGFyICpsb2NhbCA9IG1hbGxvYyhsb2NhbF9ieXRlcyk7CiAgICBpZiAoIWxvY2FsKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJyYW5rICVkOiBvdXQgb2YgbWVtb3J5XG4iLCByYW5rKTsKICAgICAgICBNUElfQWJvcnQoTVBJX0NPTU1fV09STEQsIDEpOwogICAgfQoKICAgIC8qIENvdW50cyBhbmQgZGlzcGxhY2VtZW50cyBhcmUgbmVlZGVkIGJ5IE1QSV9HYXRoZXJ2IG9uIHJhbmsgMC4gKi8KICAgIGludCAqY291bnRzID0gbWFsbG9jKChzaXplX3Qpc2l6ZSAqIHNpemVvZihpbnQpKTsKICAgIGludCAqZGlzcGxzID0gbWFsbG9jKChzaXplX3Qpc2l6ZSAqIHNpemVvZihpbnQpKTsKICAgIGlmICghY291bnRzIHx8ICFkaXNwbHMpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgInJhbmsgJWQ6IG91dCBvZiBtZW1vcnlcbiIsIHJhbmspOwogICAgICAgIE1QSV9BYm9ydChNUElfQ09NTV9XT1JMRCwgMSk7CiAgICB9CgogICAgaW50IG9mZnNldCA9IDA7CiAgICBmb3IgKGludCByID0gMDsgciA8IHNpemU7ICsrcikgewogICAgICAgIGludCByb3dzID0gYmFzZSArIChyIDwgcmVtID8gMSA6IDApOwogICAgICAgIGNvdW50c1tyXSA9IHJvd3MgKiB3aWR0aCAqIDM7CiAgICAgICAgZGlzcGxzW3JdID0gb2Zmc2V0OwogICAgICAgIG9mZnNldCArPSBjb3VudHNbcl07CiAgICB9CgogICAgdW5zaWduZWQgY2hhciAqZnVsbCA9IE5VTEw7CiAgICBpZiAocmFuayA9PSAwKSB7CiAgICAgICAgZnVsbCA9IG1hbGxvYygoc2l6ZV90KXdpZHRoICogKHNpemVfdCloZWlnaHQgKiAzdSk7CiAgICAgICAgaWYgKCFmdWxsKSB7CiAgICAgICAgICAgIGZwcmludGYoc3RkZXJyLCAicmFuayAwOiBvdXQgb2YgbWVtb3J5XG4iKTsKICAgICAgICAgICAgTVBJX0Fib3J0KE1QSV9DT01NX1dPUkxELCAxKTsKICAgICAgICB9CiAgICB9CgogICAgcHJpbnRmKCJyYW5rICVkLyVkOiB5MD0lZCByb3dzPSVkXG4iLCByYW5rLCBzaXplLCB5MCwgbG9jYWxfaCk7CgogICAgTVBJX0JhcnJpZXIoTVBJX0NPTU1fV09STEQpOwogICAgZG91YmxlIHQwID0gTVBJX1d0aW1lKCk7CgogICAgcmVuZGVyX3Jvd3MoeTAsIGxvY2FsX2gsIHdpZHRoLCBoZWlnaHQsIG1heF9pdGVyLCBsb2NhbCk7CgogICAgTVBJX0dhdGhlcnYobG9jYWwsIChpbnQpbG9jYWxfYnl0ZXMsIE1QSV9VTlNJR05FRF9DSEFSLAogICAgICAgICAgICAgICAgZnVsbCwgY291bnRzLCBkaXNwbHMsIE1QSV9VTlNJR05FRF9DSEFSLAogICAgICAgICAgICAgICAgMCwgTVBJX0NPTU1fV09STEQpOwoKICAgIGRvdWJsZSB0MSA9IE1QSV9XdGltZSgpOwoKICAgIGlmIChyYW5rID09IDApIHsKICAgICAgICB1aW50NjRfdCBoYXNoID0gZm52MWEoZnVsbCwgKHNpemVfdCl3aWR0aCAqIChzaXplX3QpaGVpZ2h0ICogM3UpOwogICAgICAgIGlmICghd3JpdGVfcHBtKG91dCwgZnVsbCwgd2lkdGgsIGhlaWdodCkpIHsKICAgICAgICAgICAgZnJlZShsb2NhbCk7CiAgICAgICAgICAgIGZyZWUoY291bnRzKTsKICAgICAgICAgICAgZnJlZShkaXNwbHMpOwogICAgICAgICAgICBmcmVlKGZ1bGwpOwogICAgICAgICAgICBNUElfQWJvcnQoTVBJX0NPTU1fV09STEQsIDEpOwogICAgICAgIH0KCiAgICAgICAgcHJpbnRmKCJNUEkrT3Blbk1QIE1hbmRlbGJyb3QgJWR4JWQgbWF4X2l0ZXI9JWQgcmFua3M9JWRcCiB0aHJlYWRzL3Jhbms9JWRcbiIsIHdpZHRoLCBoZWlnaHQsIG1heF9pdGVyLCBzaXplLCBvbXBfZ2V0X21heF90aHJlYWRzKCkpOwogICAgICAgIHByaW50ZigidGltZSAgICAgICAgOiAlLjNmIHNcbiIsIHQxIC0gdDApOwogICAgICAgIHByaW50ZigiRk5WLTFhIGhhc2ggOiAlMDE2IiBQUkl4NjQgIlxuIiwgaGFzaCk7CiAgICAgICAgcHJpbnRmKCJvdXRwdXQgICAgICA6ICVzXG4iLCBvdXQpOwogICAgfQoKCiAgICBmcmVlKGxvY2FsKTsKICAgIGZyZWUoY291bnRzKTsKICAgIGZyZWUoZGlzcGxzKTsKICAgIGZyZWUoZnVsbCk7CgogICAgTVBJX0ZpbmFsaXplKCk7CiAgICByZXR1cm4gMDsKfQo="

SERIAL_C = base64.b64decode(SERIAL_B64).decode()
DYNAMIC_C = base64.b64decode(DYNAMIC_B64).decode()
STATIC_C = DYNAMIC_C.replace(
    "#pragma omp parallel for schedule(dynamic, 8)",
    "#pragma omp parallel for schedule(static)")

(WORK / "mandelbrot_serial.c").write_text(SERIAL_C)
(WORK / "mandelbrot_mpi_omp.c").write_text(DYNAMIC_C)
(WORK / "mandelbrot_mpi_omp_static.c").write_text(STATIC_C)

for name in ["mandelbrot_serial.c", "mandelbrot_mpi_omp.c", "mandelbrot_mpi_omp_static.c"]:
    p = WORK / name
    print(f"{name:32s} {len(p.read_text().splitlines()):4d} lines  {p.stat().st_size:6d} bytes")

## 3. Build and run the original C / MPI+OpenMP project

This is equivalent to `make` + `./scripts/run.sh`, with the commands written into the notebook. If `mpicc` / `mpirun` are missing, the MPI part is skipped and the GPU section still runs.

### If MPI or OpenMP is missing

Open a Jupyter terminal and check:

```bash
which gcc mpicc mpirun
uname -m
sudo -n true && echo "sudo works" || echo "no passwordless sudo"
```

**Option A: system packages (recommended if sudo works)**

```bash
sudo apt-get update
sudo apt-get install -y build-essential libopenmpi-dev openmpi-bin
```

`build-essential` provides GCC and the OpenMP runtime (`libgomp`); `libopenmpi-dev` provides `mpicc`; `openmpi-bin` provides `mpirun`.

**Option B: conda / micromamba (no sudo)**

```bash
conda install -y -c conda-forge openmpi gcc_linux-64 gxx_linux-64
```

For an ARM container (`uname -m` returns `aarch64`), use `gcc_linux-aarch64` and `gxx_linux-aarch64` instead.

**Verify**

```bash
mpicc --version
mpirun --version
mpirun -np 2 hostname
```

Then restart the kernel and run this notebook from the top.


In [ ]:
import os, re, shutil, subprocess
from pathlib import Path

def run_cmd(cmd, env=None, timeout=180):
    print("$", " ".join(str(x) for x in cmd))
    try:
        p = subprocess.run(cmd, text=True, capture_output=True, env=env, timeout=timeout)
    except Exception as e:
        print("FAILED:", repr(e))
        return None
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p

def first_match(pattern, text, default=None):
    m = re.search(pattern, text or "")
    return m.group(1) if m else default

C_RESULTS = {}
GCC = shutil.which("gcc")
MPICC = shutil.which("mpicc")
MPIRUN = shutil.which("mpirun")

BASE_FLAGS = ["-O2", "-std=c11", "-Wall", "-Wextra", "-ffp-contract=off"]

# serial
if GCC:
    BIN_SERIAL = WORK / "mandelbrot_serial"
    p = run_cmd([GCC, *BASE_FLAGS, "-o", str(BIN_SERIAL),
                 str(WORK / "mandelbrot_serial.c"), "-lm"])
    if p and p.returncode == 0:
        p = run_cmd([str(BIN_SERIAL), "800", "600", "1000", str(WORK / "c_serial.ppm")])
        if p and p.returncode == 0:
            C_RESULTS["serial"] = {
                "time": float(first_match(r"serial time\s*:\s*([0-9.]+)", p.stdout, 0)),
                "hash": first_match(r"FNV-1a hash\s*:\s*([0-9a-f]+)", p.stdout, ""),
                "ppm": str(WORK / "c_serial.ppm"),
            }
else:
    print("gcc not found: skip C serial")

# MPI + OpenMP
if MPICC and MPIRUN and GCC:
    for label, src, out, nthreads, nranks in [
        ("static_2x2", "mandelbrot_mpi_omp_static.c", "mandelbrot_static", 2, 2),
        ("dynamic_2x2", "mandelbrot_mpi_omp.c", "mandelbrot_dynamic", 2, 2),
    ]:
        binary = WORK / out
        p = run_cmd([MPICC, *BASE_FLAGS, "-fopenmp", "-o", str(binary),
                     str(WORK / src), "-lm"])
        if not p or p.returncode != 0:
            print(f"compile failed for {label}: skip")
            continue
        env = os.environ.copy()
        env["OMP_NUM_THREADS"] = str(nthreads)
        env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
        env["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
        cmd = [MPIRUN, "--bind-to", "none", "-x", "OMP_NUM_THREADS",
               "-np", str(nranks), str(binary), "800", "600", "1000",
               str(WORK / f"c_{label}.ppm")]
        p = run_cmd(cmd, env=env)
        if p and p.returncode != 0 and "bind-to" in (p.stderr or ""):
            cmd = [MPIRUN, "-x", "OMP_NUM_THREADS", "-np", str(nranks),
                   str(binary), "800", "600", "1000", str(WORK / f"c_{label}.ppm")]
            p = run_cmd(cmd, env=env)
        if p and p.returncode == 0:
            C_RESULTS[label] = {
                "time": float(first_match(r"time\s*:\s*([0-9.]+)", p.stdout, 0)),
                "hash": first_match(r"FNV-1a hash\s*:\s*([0-9a-f]+)", p.stdout, ""),
                "ppm": str(WORK / f"c_{label}.ppm"),
            }
else:
    print("MPI toolchain missing: skip MPI+OpenMP; GPU section still runs.")
    print("Install with one of:")
    print("  sudo apt-get update && sudo apt-get install -y build-essential libopenmpi-dev openmpi-bin")
    print("  conda install -y -c conda-forge openmpi gcc_linux-64 gxx_linux-64")

print()
print("C_RESULTS =", json.dumps(C_RESULTS, indent=2))

## 4. PyTorch ROCm GPU Mandelbrot

This version maps each pixel to one GPU tensor element and updates all 800x600 pixels in a vectorised loop. `torch.cuda` is the PyTorch API used for both CUDA and ROCm; on this cluster it maps to the AMD GPU.

The default dtype is `float64`, to stay close to the C `double`. If the device cannot run it, the notebook falls back to `float32`.


In [ ]:
import time, hashlib
from pathlib import Path
import numpy as np

try:
    import torch
except Exception as e:
    raise RuntimeError("PyTorch is required for the GPU section") from e

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "CPU fallback"

print("device:", DEVICE, "|", DEVICE_NAME)
print("torch HIP:", torch.version.hip)

try:
    from PIL import Image
    PIL_OK = True
except Exception:
    PIL_OK = False
    print("Pillow not available: will still write PPM")

def mandelbrot_torch(width=800, height=600, max_iter=1000,
                     device=DEVICE, dtype=torch.float64):
    xmin, xmax = -2.10, 0.70
    yspan = (xmax - xmin) * height / width
    ymin, ymax = -0.5 * yspan, 0.5 * yspan

    x = torch.linspace(xmin, xmax, width, device=device, dtype=dtype)
    y = torch.linspace(ymax, ymin, height, device=device, dtype=dtype)
    cr = x.view(1, width).expand(height, width)
    ci = y.view(height, 1).expand(height, width)

    zr = torch.zeros_like(cr)
    zi = torch.zeros_like(ci)
    counts = torch.zeros((height, width), device=device, dtype=torch.int32)
    alive = torch.ones((height, width), device=device, dtype=torch.bool)

    for _ in range(max_iter):
        zr2 = zr * zr
        zi2 = zi * zi
        alive = alive & ((zr2 + zi2) <= 4.0)

        next_zr = zr2 - zi2 + cr
        next_zi = 2.0 * zr * zi + ci

        zr = torch.where(alive, next_zr, zr)
        zi = torch.where(alive, next_zi, zi)
        counts = counts + alive.to(torch.int32)

    return counts

def colourise(counts, max_iter=1000):
    t = torch.sqrt(counts.to(torch.float64) / 200.0).clamp(max=1.0)
    r = (255.0 * torch.clamp(2.0 * t, 0.0, 1.0)).to(torch.uint8)
    g = (255.0 * torch.clamp(2.0 * t - 0.5, 0.0, 1.0)).to(torch.uint8)
    b = (255.0 * torch.clamp(1.0 - 2.0 * t, 0.0, 1.0)).to(torch.uint8)
    rgb = torch.stack([r, g, b], dim=-1).cpu().numpy()
    rgb[counts.cpu().numpy() >= max_iter] = 0
    return rgb

def write_ppm(path, rgb):
    h, w, _ = rgb.shape
    with open(path, "wb") as f:
        f.write(f"P6\n{w} {h}\n255\n".encode("ascii"))
        f.write(rgb.astype(np.uint8).tobytes())

def fnv1a(data):
    # Use the same offset basis and prime as the C code in src/.
    # Note: this basis is 1469598103934665603, not the standard
    # FNV-1a 64-bit basis 14695981039346656037.
    h = 1469598103934665603
    for byte in data:
        h ^= int(byte)
        h = (h * 1099511628211) & 0xFFFFFFFFFFFFFFFF
    return f"{h:016x}"

def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def timed_run(width=800, height=600, max_iter=1000, dtype=torch.float64):
    sync()
    t0 = time.perf_counter()
    counts = mandelbrot_torch(width, height, max_iter, DEVICE, dtype)
    sync()
    return time.perf_counter() - t0, counts

# warm-up (small, not part of the measurement)
try:
    _ = mandelbrot_torch(320, 240, 200, DEVICE, torch.float64)
    sync()
    print("warm-up done")
except Exception as e:
    print("float64 warm-up failed, falling back to float32:", repr(e))
    DEVICE_DTYPE = torch.float32
else:
    DEVICE_DTYPE = torch.float64

GPU_TIMES = []
GPU_COUNTS = None
for i in range(3):
    dt, counts = timed_run(800, 600, 1000, DEVICE_DTYPE)
    GPU_TIMES.append(dt)
    GPU_COUNTS = counts
    print(f"GPU run {i+1}: {dt:.4f} s")

gpu_rgb = colourise(GPU_COUNTS, 1000)
gpu_ppm = WORK / "gpu_mandelbrot.ppm"
write_ppm(gpu_ppm, gpu_rgb)
gpu_bytes = gpu_ppm.read_bytes()
# The C code hashes the RGB pixels only; the PPM header is not included.
gpu_hash = fnv1a(gpu_rgb.astype(np.uint8).tobytes())
# SHA-256 is over the complete PPM file, matching sha256sum in the report.
gpu_sha = hashlib.sha256(gpu_bytes).hexdigest()

if PIL_OK:
    gpu_png = WORK / "gpu_mandelbrot.png"
    Image.fromarray(gpu_rgb).save(gpu_png)
else:
    gpu_png = None

print("GPU mean : %.4f s" % (sum(GPU_TIMES)/len(GPU_TIMES)))
print("FNV-1a   :", gpu_hash)
print("SHA-256  :", gpu_sha)
print("PPM      :", gpu_ppm)

try:
    from IPython.display import Image as IPyImage, display
    if gpu_png:
        display(IPyImage(filename=str(gpu_png)))
except Exception:
    pass

## 5. Compare results

Compare C serial, C static/dynamic MPI+OpenMP, and the GPU version. The GPU uses PyTorch floating point, which is not guaranteed to be bit-identical to C `double`; the cell reports both hashes and the pixel differences.


In [ ]:
import numpy as np
from pathlib import Path

def read_ppm_pixels(path):
    if PIL_OK:
        return np.asarray(Image.open(path).convert("RGB"))
    data = Path(path).read_bytes()
    # minimal P6 parser
    pos = 0
    tokens = []
    while len(tokens) < 4:
        while data[pos:pos+1].isspace():
            pos += 1
        if data[pos:pos+1] == b"#":
            while data[pos:pos+1] != b"\n":
                pos += 1
            continue
        start = pos
        while not data[pos:pos+1].isspace():
            pos += 1
        tokens.append(data[start:pos])
        pos += 1
    w, h = int(tokens[1]), int(tokens[2])
    return np.frombuffer(data[pos:], dtype=np.uint8).reshape(h, w, 3)

if "serial" in C_RESULTS and Path(C_RESULTS["serial"]["ppm"]).exists():
    c_rgb = read_ppm_pixels(C_RESULTS["serial"]["ppm"])
    if c_rgb.shape == gpu_rgb.shape:
        diff = np.abs(c_rgb.astype(np.int16) - gpu_rgb.astype(np.int16))
        changed = int((diff.max(axis=2) > 0).sum())
        print("pixels different from C serial:", changed, "/", c_rgb.shape[0]*c_rgb.shape[1])
        print("max channel difference       :", int(diff.max()))
        print("C serial FNV                 :", C_RESULTS["serial"]["hash"])
        print("GPU FNV                      :", gpu_hash)

rows = []
base = C_RESULTS.get("serial", {}).get("time")
for key, label in [("serial", "C serial 1x1"),
                   ("static_2x2", "C static 2x2"),
                   ("dynamic_2x2", "C dynamic 2x2")]:
    if key in C_RESULTS:
        r = C_RESULTS[key]
        sp = (base / r["time"]) if base and r["time"] else None
        rows.append((label, r["time"], sp, r["hash"]))

gpu_mean = sum(GPU_TIMES) / len(GPU_TIMES)
rows.append(("PyTorch ROCm GPU", gpu_mean,
             (base / gpu_mean) if base else None, gpu_hash))

print()
print(f"{'configuration':24s} {'time (s)':>10s} {'speed-up':>10s}  hash")
for label, t, sp, h in rows:
    sp_s = f"{sp:.2f}x" if sp else "-"
    print(f"{label:24s} {t:10.4f} {sp_s:>10s}  {h}")

try:
    import pandas as pd
    display(pd.DataFrame(rows, columns=["configuration", "time (s)", "speed-up", "FNV-1a"]))
except Exception:
    pass

try:
    import matplotlib.pyplot as plt
    labels = [r[0] for r in rows]
    vals = [r[1] for r in rows]
    plt.figure(figsize=(8, 3.5))
    plt.bar(labels, vals)
    plt.ylabel("time (s)")
    plt.title("Mandelbrot on AMD cluster")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("plot skipped:", repr(e))

summary = {
    "device": DEVICE_NAME,
    "torch": torch.__version__,
    "hip": torch.version.hip,
    "c_results": C_RESULTS,
    "gpu_times": GPU_TIMES,
    "gpu_mean": gpu_mean,
    "gpu_hash": gpu_hash,
    "gpu_sha256": gpu_sha,
}
Path(WORK / "results.json").write_text(json.dumps(summary, indent=2))
print("\nwrote", WORK / "results.json")

## 6. Comparison with the original VM (MPI + OpenMP)

The original lab report measured a controlled four-vCPU VM environment:

| Configuration | Mean time (s) | Speed-up |
|---|---:|---:|
| serial 1x1 | 0.725 | 1.00x |
| static 2x2 | 0.326 | 2.22x |
| dynamic 2x2 | 0.1833 | 3.95x |

The AMD cluster is different hardware, so a cross-environment ratio is only a
reference, not a controlled speed-up. The table below keeps two ideas separate:

- **Speed-up vs its own serial baseline** is the meaningful in-environment
  number.
- **Relative to the original VM serial** is shown only to answer "how does this
  time compare with the old result?"; it mixes CPU, GPU and compiler
  differences.

The controlled comparison is the AMD row set: C serial vs C MPI+OpenMP vs the
PyTorch ROCm GPU port, all in the same container.


In [ ]:
import json
from pathlib import Path

ORIGINAL_VM = {
    "serial": 0.725,
    "static_2x2": 0.326,
    "dynamic_2x2": 0.18333,
}

gpu_mean = sum(GPU_TIMES) / len(GPU_TIMES)
amd_serial = C_RESULTS.get("serial", {}).get("time")
amd_static = C_RESULTS.get("static_2x2", {}).get("time")
amd_dynamic = C_RESULTS.get("dynamic_2x2", {}).get("time")

if "static_2x2" not in C_RESULTS or "dynamic_2x2" not in C_RESULTS:
    print("WARNING: C static 2x2 and/or dynamic 2x2 were NOT run.")
    print("This happens when mpicc or mpirun is missing; the notebook skips MPI.")
    print("Install OpenMPI, restart the kernel, then Run All from the first cell.")
    print()

def fmt_time(t):
    return f"{t:.4f}" if t is not None else "n/a"

def fmt_ratio(r):
    return f"{r:.2f}x" if r is not None else "n/a"

rows = []
rows.append(("Original 4-vCPU VM", "serial 1x1", ORIGINAL_VM["serial"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["serial"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["serial"]))
rows.append(("Original 4-vCPU VM", "static 2x2", ORIGINAL_VM["static_2x2"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["static_2x2"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["static_2x2"]))
rows.append(("Original 4-vCPU VM", "dynamic 2x2", ORIGINAL_VM["dynamic_2x2"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["dynamic_2x2"],
             ORIGINAL_VM["serial"] / ORIGINAL_VM["dynamic_2x2"]))

if amd_serial is not None:
    rows.append(("AMD cluster", "C serial", amd_serial, 1.0,
                 ORIGINAL_VM["serial"] / amd_serial))
if amd_static is not None and amd_serial is not None:
    rows.append(("AMD cluster", "C static 2x2", amd_static,
                 amd_serial / amd_static, ORIGINAL_VM["serial"] / amd_static))
if amd_dynamic is not None and amd_serial is not None:
    rows.append(("AMD cluster", "C dynamic 2x2", amd_dynamic,
                 amd_serial / amd_dynamic, ORIGINAL_VM["serial"] / amd_dynamic))

rows.append(("AMD cluster", "PyTorch ROCm GPU", gpu_mean,
             (amd_serial / gpu_mean) if amd_serial else None,
             ORIGINAL_VM["serial"] / gpu_mean))

print(f"{'environment':20s} {'configuration':18s} {'time (s)':>10s} "
      f"{'vs own serial':>14s} {'vs orig. serial':>16s}")
for env, cfg, t, own, ref in rows:
    print(f"{env:20s} {cfg:18s} {fmt_time(t):>10s} "
          f"{fmt_ratio(own):>14s} {fmt_ratio(ref):>16s}")

print()
if amd_serial:
    print("GPU vs AMD C serial      : %.2fx" % (amd_serial / gpu_mean))
if amd_dynamic:
    print("GPU vs AMD C dynamic 2x2 : %.2fx" % (amd_dynamic / gpu_mean))
print("GPU vs original VM dynamic 2x2 (cross-environment reference): %.2fx"
      % (ORIGINAL_VM["dynamic_2x2"] / gpu_mean))

try:
    import pandas as pd
    display(pd.DataFrame(rows, columns=[
        "environment", "configuration", "time (s)",
        "speed-up vs own serial", "relative to original VM serial"]))
except Exception:
    pass

try:
    import matplotlib.pyplot as plt
    labels = ["orig. serial", "orig. dynamic 2x2", "AMD C serial",
              "AMD C dynamic 2x2", "AMD GPU"]
    values = [ORIGINAL_VM["serial"], ORIGINAL_VM["dynamic_2x2"],
              amd_serial, amd_dynamic, gpu_mean]
    plt.figure(figsize=(9, 3.5))
    plt.bar(labels, values)
    plt.ylabel("time (s)")
    plt.title("Different hardware: reference comparison, not a controlled speed-up")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("plot skipped:", repr(e))

summary["original_vm"] = ORIGINAL_VM
summary["comparison"] = {
    "amd_c_serial": amd_serial,
    "amd_c_static_2x2": amd_static,
    "amd_c_dynamic_2x2": amd_dynamic,
    "amd_gpu_mean": gpu_mean,
    "gpu_vs_amd_serial": (amd_serial / gpu_mean) if amd_serial else None,
    "gpu_vs_amd_dynamic_2x2": (amd_dynamic / gpu_mean) if amd_dynamic else None,
    "gpu_vs_original_dynamic_2x2": ORIGINAL_VM["dynamic_2x2"] / gpu_mean,
}
Path(WORK / "results.json").write_text(json.dumps(summary, indent=2))
print("updated", WORK / "results.json")


## 7. Conclusions and caveats

- If `mpicc` / `mpirun` exist, the notebook runs C serial, static 2x2 and dynamic 2x2. If not, it runs C serial and the GPU version only.
- The GPU version uses PyTorch ROCm. `torch.cuda` maps to HIP on AMD and does not need a CUDA driver.
- With `float64`, the GPU result is closer to C `double`, but PyTorch and compilers may still use FMA; boundary pixels can differ by one iteration, so FNV/SHA are not guaranteed to match. The pixel-difference count reports the actual difference.
- The C hash also depends on the compiler, libm and CPU architecture. `92193b63bbd42a23` is the result from the original VM; the AMD container may produce a different hash. Compare hashes within the same environment.
- The MPI part in this Jupyter container is single-node multi-process. The multi-node result is the two-node check in the report.
- The GPU version is the ROCm port and performance experiment; the C version remains the reference for the lab's MPI+OpenMP result.

Full project:

https://github.com/YohtHill/mandelbrot-mpi-openmp

Original GitCode:

https://gitcode.com/JiakunPi/mandelbrot-mpi-openmp
